# Exploring the programs

`chart programs` leaves its tables in `programs/` under `data_output_dir`. This notebook shows what each holds and a few ways to read them, following the analyses in cellmapp's notebooks 04 and 05. Set `CONFIG` to your config file and run it from the top.

| Table | One row per | Holds |
|---|---|---|
| `matrix.parquet` | gene | its signed effect on each archetype, for the genes and archetypes with any q-value below the threshold |
| `loadings.parquet` | archetype | its loading on each program |
| `scores.parquet` | gene | its score on each program |
| `modules.parquet` | gene | its module, and how firmly it belongs there |
| `consensus.parquet` | gene | how often it was grouped with each other gene |
| `cell_scores.parquet` | cell | its score on each program |
| `top_cells.parquet` | program and rank | the example cells, with their bounding boxes |

Gene and cell scores are on the same scale. A gene's `margin` is how much more often it was grouped with its own module than with the next closest one; below 0.15 it is called `weak`.

In [ ]:
CONFIG = 'my_screen.yaml'

MIN_MODULE_SIZE = 3    # smaller modules are left out of the module summaries
STRONG_GENES = None    # e.g. 1.2 keeps only genes whose largest effect is above it, as notebook 04 did
MODULE = None          # a module to look at in detail, such as 'M3'; None takes the largest
PROGRAM = 'P1'         # a program to look at cell by cell

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from chart.io.config import load_archetypes_config, load_programs_config
from chart.io.tables import load_table

directory = load_programs_config(CONFIG).outputs_path
matrix, loadings, scores, modules = (load_table(directory, name)
                                     for name in ('matrix', 'loadings', 'scores', 'modules'))
programs = list(loadings.columns)

## What each program is made of

A program loads on a few archetypes: positively on those its genes put more weight in, negatively on those they put less in. These are its largest loadings, and the genes that score highest and lowest on it.

In [ ]:
def largest(series, n):
    series = series[series != 0]
    return ', '.join(f'{name} {value:+.2f}' for name, value in
                     series.sort_values(key=np.abs, ascending=False).head(n).items())

pd.DataFrame({'archetypes': {p: largest(loadings[p], 6) for p in programs},
              'highest genes': {p: ', '.join(scores[p].nlargest(5).index) for p in programs},
              'lowest genes': {p: ', '.join(scores[p].nsmallest(5).index) for p in programs}})

## Modules

Modules with at least `MIN_MODULE_SIZE` genes are named M1, M2 and so on, in the order of their numbers in `modules.parquet`; the rest are left out of the summaries below.

In [ ]:
genes = matrix.index
if STRONG_GENES is not None:
    genes = genes[matrix.abs().max(axis=1) > STRONG_GENES]
module = modules.loc[genes, 'module']
sizes = module.value_counts()
names = {m: f'M{i + 1}' for i, m in enumerate(sorted(sizes.index[sizes >= MIN_MODULE_SIZE]))}
named = module[module.isin(list(names))].map(names)
order = list(names.values())
print(f'{len(names)} modules of {MIN_MODULE_SIZE} or more genes; '
      f'{len(module) - len(named)} genes in smaller ones are left out')
modules.loc[named.index].assign(name=named).sort_values(['module', 'margin'], ascending=[True, False])

## Module × program scores

Each module's median score on each program. Each program is first divided by its spread across the genes, so that programs with large scores do not drown out the rest.

In [ ]:
scaled = scores.loc[named.index] / scores.loc[genes].std().replace(0, 1)
medians = scaled.groupby(named).median().loc[order]
fig, ax = plt.subplots(figsize=(0.6 * len(programs) + 2, 0.4 * len(order) + 1.5))
limit = np.abs(medians.to_numpy()).max()
sns.heatmap(medians, ax=ax, cmap='RdBu_r', center=0, vmin=-limit, vmax=limit,
            annot=True, fmt='.1f', annot_kws={'fontsize': 7},
            yticklabels=[f'{m} (n={(named == m).sum()})' for m in order],
            cbar_kws={'label': 'median score / spread'})
ax.set_xlabel('')
plt.show()

## Which programs explain each module

Each gene's effects, its row of `matrix.parquet`, are rebuilt as one piece per program: its score times the program's loadings. A program's contribution to a gene is the share of the gene's effects its piece accounts for. Summed over the programs, the contributions give the share the programs explain together; the rest is left unexplained. Below are each module's median contributions.

In [ ]:
W = matrix.loc[named.index, loadings.index].to_numpy()
pieces = np.einsum('ik,ak->ika', scores.loc[named.index, programs].to_numpy(), loadings.to_numpy())
norm = (W ** 2).sum(axis=1)
norm = np.where(norm > 0, norm, 1.0)
contributions = pd.DataFrame(np.einsum('ia,ika->ik', W, pieces) / norm[:, None],
                             index=named.index, columns=programs)
explained = pd.Series(1 - ((W - pieces.sum(axis=1)) ** 2).sum(axis=1) / norm,
                      index=named.index, name='share explained')
by_module = contributions.groupby(named).median().loc[order]

palette = dict(zip(programs, sns.color_palette('tab20b', len(programs))))
x = np.arange(len(order))
up, down = np.zeros(len(order)), np.zeros(len(order))
fig, ax = plt.subplots(figsize=(max(8, 0.5 * len(order) + 3), 5))
for program in by_module.abs().sum().sort_values(ascending=False).index:
    values = by_module[program].to_numpy()
    ax.bar(x, np.clip(values, 0, None), bottom=up, color=palette[program], label=program)
    ax.bar(x, np.clip(values, None, 0), bottom=down, color=palette[program])
    up += np.clip(values, 0, None)
    down += np.clip(values, None, 0)
ax.axhline(0, color='black', lw=1)
ax.axhline(1, color='black', lw=0.6, ls=':')
ax.set_xticks(x, order)
ax.set_ylabel('median contribution per gene')
ax.legend(bbox_to_anchor=(1.01, 1), loc='upper left', frameon=False, fontsize=8, title='Program')
plt.show()

## One module in detail

Each gene's contribution from each program, with the median as a bar, and how much of each gene's effects the programs explain.

In [ ]:
name = MODULE or named.value_counts().index[0]
members = named.index[named == name]
print(f'{name}: {", ".join(members)}')

fig, (left, right) = plt.subplots(1, 2, figsize=(14, 4), gridspec_kw={'width_ratios': [3, 1]})
detail = contributions.loc[members]
left.bar(programs, detail.median(), color='lightgrey')
jitter = np.random.default_rng(0).uniform(-0.2, 0.2, size=len(members))
for k, program in enumerate(programs):
    left.scatter(k + jitter, detail[program], s=8, color='black')
left.axhline(0, color='black', lw=0.6)
left.set_ylabel('contribution')
explained.loc[members].sort_values().plot.barh(ax=right, color='grey')
right.set_xlabel('share explained')
plt.tight_layout()
plt.show()

## Cells

`cell_scores.parquet` scores every cell on every program. It has one row per cell, so it can be large.

In [ ]:
cells = load_table(directory, 'cell_scores').set_index(['Well', 'Label', 'Guide', 'Gene'])
ncol = 6
fig, axes = plt.subplots(int(np.ceil(len(programs) / ncol)), ncol,
                         figsize=(2.6 * ncol, 2 * np.ceil(len(programs) / ncol)), squeeze=False)
for ax, program in zip(axes.ravel(), programs):
    ax.hist(cells[program], bins=80, color='#4472c4')
    ax.set_title(f'{program} (median {cells[program].median():.2f})', fontsize=9)
    ax.tick_params(labelsize=7)
for ax in axes.ravel()[len(programs):]:
    ax.axis('off')
plt.tight_layout()
plt.show()

## Where a program's top and bottom cells sit

For `PROGRAM`'s largest loadings, the archetype weights of its top and bottom 0.5% of cells against a random sample of cells. Where the program loads positively, its top cells should carry more of that archetype's weight than most cells.

In [ ]:
archetypes = load_archetypes_config(CONFIG)
chosen = loadings[PROGRAM][loadings[PROGRAM] != 0].sort_values(key=np.abs, ascending=False).head(8)
raw = {}
for channel in sorted({a.rsplit('_', 1)[0] for a in chosen.index}):
    weights = pd.read_parquet(archetypes.weights_path(channel)).set_index(['Well', 'Label', 'Guide', 'Gene'])
    raw.update({f'{channel}_{c}': weights[c] for c in weights.columns})
raw = pd.DataFrame(raw)

score = cells[PROGRAM]
groups = {'random cells': (score.sample(min(5000, len(score)), random_state=0).index, '#888888'),
          'top 0.5%': (score.index[score >= np.percentile(score, 99.5)], '#ba0025'),
          'bottom 0.5%': (score.index[score <= np.percentile(score, 0.5)], '#0076b6')}
fig, axes = plt.subplots(1, len(chosen), figsize=(3 * len(chosen), 2.6), squeeze=False)
for ax, (archetype, loading) in zip(axes[0], chosen.items()):
    for label, (index, color) in groups.items():
        values = raw[archetype].reindex(index).dropna()
        if values.nunique() > 1:
            sns.kdeplot(values, ax=ax, color=color, cut=0, label=label,
                        fill=label == 'random cells', linewidth=0 if label == 'random cells' else 1.8)
    ax.set_title(f'{archetype}\nloading {loading:+.2f}', fontsize=9)
    ax.set_xlabel('weight')
    ax.tick_params(axis='y', left=False, labelleft=False)
    ax.set_ylabel('')
axes[0, 0].legend(fontsize=7, frameon=False)
plt.tight_layout()
plt.show()

## Example cells

`top_cells.parquet` holds 20 example cells per program: of its 100 highest-scoring cells, those that score most above their average on the other programs. Their bounding boxes are there to crop them from the images.

In [ ]:
examples = load_table(directory, 'top_cells')
examples[examples.Program == PROGRAM]